### This code reads in a dataset of phenology observations, interpolates ERA5 data to the given locations for a growing season, and saves the file as a CSV for use in modelling.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import scipy.optimize
import scipy.stats
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from sklearn import tree
from sklearn.model_selection import train_test_split
import copy
import calendar  

import plotting
import dataset_fctns
import modelling_fctns
import seaborn as sns
#from dwd_phenpy import Phenology_set

In [2]:
phen = pd.read_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\maize_phenology2_full_june_2025.csv')

In [2]:
class Phenology_set:

    phase_names = pd.read_csv("https://opendata.dwd.de/climate_environment/CDC/help/PH_Beschreibung_Phase.txt", encoding = "latin1", engine='python', sep = r';\s+|;\t+|;\s+\t+|;\t+\s+|;|\s+;|\t+;|\s+\t+;|\t+\s+;')
    station_data1 = pd.read_csv("https://opendata.dwd.de/climate_environment/CDC/help/PH_Beschreibung_Phaenologie_Stationen_Jahresmelder.txt",sep = ";\s+|;\t+|;\s+\t+|;\t+\s+|;|\s+;|\t+;|\s+\t+;|\t+\s+;", encoding='cp1252', on_bad_lines='skip')
    station_data2 = pd.read_csv("https://opendata.dwd.de/climate_environment/CDC/help/PH_Beschreibung_Phaenologie_Stationen_Sofortmelder.txt",sep = ";\s+|;\t+|;\s+\t+|;\t+\s+|;|\s+;|\t+;|\s+\t+;|\t+\s+;", encoding='cp1252', on_bad_lines='skip')
    station_data = pd.concat([station_data1, station_data2], axis=0)
    def __init__(self, address, raw = False, dwd_data = True, full_ds = False, already_located = False):
        if already_located:
            self.identifying_subset = ['Referenzjahr', 'Stations_id', 'lat', 'lon']
        else:
            self.identifying_subset = ['Referenzjahr', 'Stations_id']
        if raw:
            self.phen_data = pd.read_csv(address, encoding = "latin1", engine='python', sep = r';\s+|;\t+|;\s+\t+|;\t+\s+|;|\s+;|\t+;|\s+\t+;|\t+\s+;')
        else:
            self.phen_data = pd.read_csv(address)
        ## CONVERT DATE TO DATETIME ##
        if dwd_data:
            self.phen_data['Eintrittsdatum'] = pd.to_datetime(self.phen_data['Eintrittsdatum'], format = '%Y%m%d')
            if full_ds:
                self.phen_data = self.phen_data.loc[(self.phen_data['Qualitaetsniveau'] == 10) | ((self.phen_data['Qualitaetsniveau'] == 7)*(self.phen_data['Reporting method'] == 'immediate'))]
            else:
                self.phen_data = self.phen_data.loc[(self.phen_data['Qualitaetsniveau'] == 10)]
            if not(already_located):
                self.add_locations()
        self.phase_list = [] #list of phases to consider
        #print(self.phen_data['Qualitaetsniveau'].values)
        self.T_mean = ''
        self.GDD_driver_data = ''
        self.ordered = False
        self.first_input_array = True

    ### Functions for sorting out dataset ###
    def drop_columns(self, drop_list):
        for drop_name in drop_list:
            try:
                self.phen_data = self.phen_data.drop(drop_name, axis = 1)
            except:
                print(f'Column {drop_name} not found')
                continue
    
    def phase_order_name(self, stage_order): #[10, 12, 67, 65, 5, 6, 19, 20, 21, 24, ]
        self.phen_data['Order of phase'] = np.nan
        self.phen_data['Name of phase'] = ''
        for i, phaseid in enumerate(stage_order):
            if len(self.phase_names['Phase_englisch'][self.phase_names['Phase_ID'] == str(phaseid)]) != 0:
                #print(i, phaseid)
                self.phen_data.loc[self.phen_data['Phase_id'] == phaseid, 'Order of phase'] = i
                self.phen_data.loc[self.phen_data['Phase_id'] == phaseid, 'Name of phase'] = dataset_fctns.get_phase_name(phaseid, self.phase_names)
        self.order_phen_dataset()

    def order_phen_dataset(self):
        ## SORT BY TIME ##
        if not(np.isin('Order of phase', self.phen_data.columns)):
            print('Get phase order and names first')
        else:
            self.phen_data.sort_values(by = ['Stations_id', 'Referenzjahr', 'Eintrittsdatum', 'Order of phase'])
            self.ordered = True
    
    def get_time_to_next_stage(self):
        #Note phen_data must be time and station ordered. Only plots time to next stage - naive as doesn't consider missing phases.
        if self.ordered:
            ## CALCULATE TIME TO NEXT STAGE ##
            self.phen_data['Time to next stage'] = self.phen_data['Eintrittsdatum'].shift(-1) - self.phen_data['Eintrittsdatum']
            self.phen_data['Next stage name'] = self.phen_data['Name of phase'].shift(-1)
            ## EXCLUDE CHANGES IN STATION ##
            self.phen_data.loc[self.phen_data['Stations_id'] != self.phen_data['Stations_id'].shift(-1), 'Time to next stage'] = np.nan
            self.phen_data.loc[self.phen_data['Stations_id'] != self.phen_data['Stations_id'].shift(-1), 'Next stage name'] = np.nan
        else:
            print('Order dataset so I can get time to next stage')

    def add_locations(self):
        self.phen_data = dataset_fctns.get_station_locations(self.phen_data, self.station_data)
        #LAT, LON = dataset_fctns.get_station_locations(self.phen_data, self.station_data)
        #self.phen_data['lat'] = LAT
        #self.phen_data['lon'] = LON
        #self.phen_data['lat'] = self.phen_data['lat'].map(lambda x: x[0] if isinstance(x, np.float64) == False else x)
        #self.phen_data['lon'] = self.phen_data['lon'].map(lambda x: x[0] if isinstance(x, np.float64) == False else x)
    ### Functions for applying GDD model ###
    def get_mean_T(self, T_address):
        self.T_mean = xr.open_dataset(T_address)

    def index_time_from_emergence_day(self):
        i_day = self.GDD_driver_data['emergence_dates'].values.copy()
        i_daysofyear = np.array([i_day + np.timedelta64(12, 'h') + np.timedelta64(day_of_year, 'D') for day_of_year in range(366)])
        time_indexer = xr.DataArray(i_daysofyear, dims=[ "time", 'modelpoint'])
        self.GDD_driver_data = self.GDD_driver_data.sel(time=time_indexer, method='nearest')

    def align_emergence_obs_with_driver_data(self):
        ## Make sure we are comparing to observations where we have the driver data;
        #1. Align the times - need to check as it might run for some days then go off the end.
        #self.just_emergence = self.just_emergence.where(self.just_emergence['Referenzjahr'] <= 2024)
        ## Make sure all elements are in the driver data
        self.just_emergence = self.just_emergence.loc[np.isin(self.just_emergence['Eintrittsdatum'] + np.timedelta64(12, 'h'), self.GDD_driver_data['time'])]
        self.just_emergence = self.just_emergence.loc[np.isin(self.just_emergence['Stations_id'], self.GDD_driver_data['Stations_id'])]
        self.just_emergence = self.just_emergence.dropna()
    
    def get_unique_xy_station(self, x_coords, y_coords, station_ids):
        unique_values = np.unique(np.stack([x_coords, y_coords, station_ids]), axis = 1)
        return unique_values[0, :], unique_values[1, :], unique_values[2, :]

    def make_input_array(self, epsg_num = 3035, latlon_proj = False, start_year = 2001):
        self.latlon_proj = latlon_proj
        ## Puts pandas phenological frame into driver xarray and aligns the two
        #self.just_emergence = self.phen_data.where(self.phen_data['Name of phase'] == 'beginning of emergence').dropna()
        ## For now just do data after 2005 to save time
        if self.first_input_array:
            self.obs_for_GDD = self.phen_data.where(self.phen_data['Eintrittsdatum'] >= np.datetime64(f'{start_year}-01-01')).dropna(how='all')
            x_coords = np.float64(self.obs_for_GDD['lon'].values)
            y_coords = np.float64(self.obs_for_GDD['lat'].values)
            station_ids = np.int64(self.obs_for_GDD['Stations_id'].values)
            #print(y_coords[y_coords.dtype == str])#, y_coords + station_ids)
            print(len(np.unique(station_ids)))
            x_unique, y_unique, stations = self.get_unique_xy_station(x_coords, y_coords, station_ids)
            print(len(stations))
            #Makes an array to put into GDD model
            print('project to new coords')
            self.stations = stations
            if not(latlon_proj):
                x_epsg, y_epsg = dataset_fctns.latlon_to_projection(x_unique, y_unique, epsg_num = epsg_num)
                self.x_driver_proj = x_epsg
                self.y_driver_proj = y_epsg
            else:
                self.x_driver_proj = x_unique
                self.y_driver_proj = y_unique
            print('interpolate driver to station locations')
            # Working in xarray (not pandas) after this point:
            #print('Latlonproj:', not(latlon_proj))
            self.GDD_driver_data = dataset_fctns.interpolate_xy(self.x_driver_proj, self.y_driver_proj, self.T_mean, xy=not(latlon_proj))
            self.GDD_driver_data = self.GDD_driver_data.assign_coords(Stations_id=("modelpoint", self.stations))
            if not(latlon_proj):
                self.GDD_driver_data = self.GDD_driver_data.drop_dims('bnds')
            self.GDD_driver_data = self.GDD_driver_data.set_xindex(['Stations_id'])
            self.first_input_array = False
        else: 
            new_GDD_driver_data = dataset_fctns.interpolate_xy(self.x_driver_proj, self.y_driver_proj, self.T_mean, xy=not(latlon_proj))
            new_GDD_driver_data = new_GDD_driver_data.assign_coords(Stations_id=("modelpoint", self.stations))
            if not(latlon_proj):
                new_GDD_driver_data = self.GDD_driver_data.drop_dims('bnds')
            new_GDD_driver_data = new_GDD_driver_data.set_xindex(['Stations_id'])
            self.GDD_driver_data = xr.concat([self.GDD_driver_data, new_GDD_driver_data], dim='time')
            self.GDD_driver_data = self.GDD_driver_data.sortby('time')

    def dev_under_response(self, response, driver_variable, maturity_t_dev):
        # Response is the rate response to driver values. Driver values are the input to this response. Maturity_t_dev is the t_dev value where we should stop running.
        self.obs_for_GDD = self.obs_for_GDD.where(self.obs_for_GDD['Referenzjahr'] <= 2023)
        ## Make the indexer to extract things at the right time.
        #self.align_emergence_obs_with_driver_data()
        self.obs_for_GDD = dataset_fctns.add_SOS_to_df(self.obs_for_GDD)
        self.obs_for_GDD['WC SOS date'] = pd.to_datetime(self.obs_for_GDD['Referenzjahr'], format='%Y') + pd.to_timedelta(self.obs_for_GDD['SOS'], 'D')
        time_station = xr.Dataset.from_dataframe(self.obs_for_GDD[['Stations_id', 'WC SOS date']])
        time_station = time_station.rename({'index':'Emergence observation', 'WC SOS date':'time'})
        if not(self.latlon_proj):
            time_station['time'] += np.timedelta64(12, 'h')
        ## Initiate development time storage object.
        t_dev = np.zeros(time_station.sizes['Emergence observation']) #Continuous development time. When this passes through some thresholds then have change in phase.
        dev_time_series = [t_dev.copy()]
        ## Make sure driver dataset uses station id to index this dimension
        try:
            self.GDD_driver_data = self.GDD_driver_data.set_xindex(['Stations_id'])
        except:
            print('Couldn\'t reset index for station')
        #Run model
        for day in range(300):
            #print(day)
            driver_values = self.GDD_driver_data.sel(time_station)[driver_variable].values 
            t_dev += response(driver_values, t_dev)
            dev_time_series.append(t_dev.copy())
            time_station['time'] += np.timedelta64(1, 'D')
        dev_time_series.append(self.obs_for_GDD['Eintrittsdatum'].values.astype('datetime64[Y]'))
        dev_time_series.append(self.obs_for_GDD['Stations_id'].values)
        self.model_dev_time_series = np.array(dev_time_series)
        self.GDD_driver_data['Development Time'] = (('days from emergence', 'Emergence observation'), self.model_dev_time_series)

    def get_phase_dates(self, thresholds):
        column_names = np.concatenate([np.array(thresholds), ['Referenzjahr'], ['Stations_id']])
        self.phase_dates_array = np.zeros((len(thresholds), self.model_dev_time_series.shape[1]))
        for obs_index in range(self.model_dev_time_series.shape[1]):
            self.phase_dates_array[:, obs_index] = np.digitize(thresholds, self.model_dev_time_series[:-2, obs_index].astype(np.float64))
        self.phase_dates_array = np.concatenate([self.phase_dates_array, [pd.to_datetime(self.model_dev_time_series[-2]).year], [self.model_dev_time_series[-1]]], axis=0)
        self.phase_dates_array = pd.DataFrame(self.phase_dates_array.T, columns = column_names)
        self.phase_dates_array.set_index(self.identifying_subset)#['Referenzjahr', 'Stations_id'])
        self.phase_dates_calculated = True
        
        #Note that the thresholds are NOT the bins for numpy digitize!
    
    ## Functions for evaluation ##
    def get_observed_dataset(self, winter_sowing = False, count_from_SOS = True, pd_SOS = False, use_metadata = False):
        if pd_SOS:
            SOS_name = 'Planting date'
        else:
            SOS_name = 'WC SOS date'
        if use_metadata:
            self.metadata_cols = ['Objekt_id', 'Reporting method', 'Historic or recent', 'ERNTEVERFAHREN_ID', 'ERNTEVERFAHREN', 'SORTE_ID', 'SORTE', 'SILOREIFEZAHL', 'KOERNERREIFEZAHL']
        else:
            self.metadata_cols = []
        if count_from_SOS:
            self.phen_data = dataset_fctns.add_SOS_to_df(self.phen_data)
            self.ds_observed = self.phen_data[['Stations_id', 'Referenzjahr', 'lat', 'lon', 'SOS'] + self.metadata_cols].drop_duplicates()
            if pd_SOS:
                just_phase = self.phen_data.loc[self.phen_data['Name of phase'] == 'beginning of tilling sowing drilling']
                self.ds_observed = self.ds_observed.merge(just_phase[['Eintrittsdatum'] + self.identifying_subset], how = 'left', on = self.identifying_subset).rename(columns={'Eintrittsdatum': SOS_name})
            else:
                self.phen_data[SOS_name] = pd.to_datetime(self.phen_data['Referenzjahr'], format='%Y') + pd.to_timedelta(self.phen_data['SOS'], 'D')
                self.ds_observed[SOS_name] = pd.to_datetime(self.ds_observed['Referenzjahr'], format='%Y') + pd.to_timedelta(self.ds_observed['SOS'], 'D')
            for phase in self.phase_list:
                just_phase = self.phen_data.loc[self.phen_data['Name of phase'] == phase]
                just_phase= just_phase.assign(**{f'observed time to {phase}': just_phase['Eintrittsdatum']})
                self.ds_observed = self.ds_observed.merge(just_phase[[f'observed time to {phase}'] + self.identifying_subset], how = 'left', on = self.identifying_subset)
                self.ds_observed[f'observed time to {phase}'] = self.ds_observed[f'observed time to {phase}'] - self.ds_observed[SOS_name]
        else:
            observed_to_first_stage = dataset_fctns.time_stage_to_stage(self.phen_data, 'beginning of emergence', self.phase_list[0], winter_sowing=winter_sowing).dropna()
            self.ds_observed = pd.DataFrame({f'observed time to {self.phase_list[0]}': observed_to_first_stage})
            for phase in self.phase_list[1:]:
                self.ds_observed[f'observed time to {phase}'] = dataset_fctns.time_stage_to_stage(self.phen_data, 'beginning of emergence', phase, winter_sowing=winter_sowing).dropna()
            self.ds_observed = self.ds_observed.reset_index()
            self.ds_observed = dataset_fctns.get_station_locations(self.ds_observed, self.station_data)
            self.ds_observed = self.ds_observed.merge(self.obs_for_GDD[['Eintrittsdatum'] + self.identifying_subset], how = 'outer', on=self.identifying_subset).rename(columns={'Eintrittsdatum':'emergence date'})
        #self.ds_observed = self.ds_observed.set_index(['Referenzjahr', 'Stations_id'])
        #self.ds_observed = pd.concat([self.just_emergence.set_index(['Referenzjahr', 'Stations_id'], inplace=False)['Eintrittsdatum'], self.ds_observed], axis=1).rename(columns={'Eintrittsdatum':'emergence date'})
        #LAT, LON = dataset_fctns.get_station_locations(self.ds_observed, self.station_data)
        #self.ds_observed['lat'] = LAT
        #self.ds_observed['lon'] = LON
        #self.ds_observed['lat'] = self.ds_observed['lat'].map(lambda x: x[0] if isinstance(x, np.float64) == False else x)
        #self.ds_observed['lon'] = self.ds_observed['lon'].map(lambda x: x[0] if isinstance(x, np.float64) == False else x)
    
    def compare_modelled_observed(self):
        self.ds_modelled_observed = pd.merge(self.ds_observed, self.phase_dates_array, how='outer', on=self.identifying_subset)#['Referenzjahr', 'Stations_id'])

    def get_X_y_for_ML2(self, driver_variable = 't2m', start_year = 2001, end_year = 2025, drop_columns = ['number', 'lon', 'lat', 'observation'], SOS_name = 'WC SOS date', numdays = 220):
        self.observations_to_use = self.ds_observed[[SOS_name] + self.identifying_subset + self.metadata_cols].loc[(self.ds_observed[SOS_name] >= np.datetime64(f'{start_year}-01-01'))*(self.ds_observed[SOS_name] <= np.datetime64(f'{end_year + 1}-01-01'))].dropna(how='all').drop_duplicates(subset = self.identifying_subset)#['Referenzjahr', 'Stations_id'])
        #print(self.observations_to_use, self.observations_to_use.drop_duplicates(subset = ['Stations_id', 'Planting date']))
        #self.observations_to_use  = self.observations_to_use.loc[self.observations_to_use[SOS_name].dt.year == self.observations_to_use['Referenzjahr']]
        print(len(self.observations_to_use))#, self.observations_to_use.drop_duplicates(['Referenzjahr', 'Stations_id']))
        # make an indexing array to pull values from the array of temperatures
        time_station = xr.Dataset.from_dataframe(self.observations_to_use)
        time_station = time_station.rename({'index':'observation', SOS_name:'time'})
        #print(time_station)
        if not(self.latlon_proj):
            time_station['time'] += np.timedelta64(12, 'h')# - np.timedelta64(60, 'D')

        ## Initiate development time storage object - a list with a value for all the stations, that will change over time and be stored in a list.
        t_dev = np.zeros(time_station.sizes['observation']) #Continuous development time. When this passes through some thresholds then have change in phase.
        dev_time_series = [t_dev.copy()]
        ## Make sure driver dataset uses station id to index this dimension
        try:
            self.GDD_driver_data = self.GDD_driver_data.set_xindex(['Stations_id'])
        except:
            print('Couldn\'t reset index for station')
        #print(time_station)
        #print(time_station[['observation', 'time']].drop_duplicates('observation'))
        #Run model
        first = True
        for day in range(numdays):
            # Pull values for temperature out of data frame
            print(day)
            #print(time_station['Referenzjahr'])
            driver_values = self.GDD_driver_data.sel(time_station[['Stations_id', 'time']])#[driver_variable]#.values
            for variable_name in self.identifying_subset:
                driver_values[variable_name] = time_station[variable_name]
            
            driver_frame_at_day = driver_values[[driver_variable, 'time'] + self.identifying_subset].to_pandas().reset_index().drop(drop_columns, axis=1)
            if first:
                years = driver_frame_at_day['time'].dt.year
                first = False
            #driver_frame_at_day['Referenzjahr'] = years #driver_frame_at_day['time'].dt.year
            #driver_frame_at_day.loc[driver_frame_at_day['time'].dt.dayofyear < 80, 'Referenzjahr'] = driver_frame_at_day.loc[driver_frame_at_day['time'].dt.dayofyear < 50, 'time'] .dt.year - 1
            #print(driver_frame_at_day)
            driver_frame_at_day = driver_frame_at_day.drop('time', axis=1)
            driver_frame_at_day = driver_frame_at_day.rename(columns = {driver_variable:f'{driver_variable} at day {day}'})
            #print(driver_frame_at_day)
            #print(driver_frame_at_day.drop_duplicates(subset = ['Referenzjahr', 'Stations_id']))
            print(len(self.observations_to_use))
            #print(driver_frame_at_day.columns.to_list())
            #print(self.observations_to_use.columns.to_list())
            #print(self.observations_to_use.columns.to_list())
            self.observations_to_use = self.observations_to_use.merge(driver_frame_at_day, on=self.identifying_subset).dropna(how='all')
            time_station['time'] += np.timedelta64(1, 'D')
        self.driver_frame_for_ML = self.observations_to_use.merge(self.ds_observed[self.identifying_subset + [f'observed time to {phase}' for phase in self.phase_list]]).drop_duplicates(subset = self.identifying_subset)#['Referenzjahr', 'Stations_id'])

    def get_X_y_for_ML(self, driver_variable = 'tas', predictor_days = 200, cumulative = False, thinning_parameter = 1, start_year = 2020, end_year = 2023):
        self.just_emergence = dataset_fctns.add_EOS_to_df(self.just_emergence)
        self.just_emergence = dataset_fctns.add_SOS_to_df(self.just_emergence)
        self.just_emergence['WC SOS date'] = pd.to_datetime(self.just_emergence['Referenzjahr'], format='%Y') + pd.to_timedelta(self.just_emergence['SOS'], 'D')
        self.just_emergence['SOS'] = pd.to_timedelta(self.just_emergence['SOS'], 'D')
        time_station = xr.Dataset.from_dataframe(self.just_emergence[['Stations_id', 'SOS']].drop_duplicates()) #, 'Referenzjahr'
        time_station = time_station.set_coords('Stations_id').set_xindex(['Stations_id'])
        time_station = time_station.drop_vars('index')
        time_station = time_station.expand_dims(dim={'time':pd.to_timedelta(np.arange(0, predictor_days), 'D')})
        time_station = time_station.expand_dims(dim={'Referenzjahr':pd.date_range(f'{start_year}-01-01', periods = end_year - start_year, freq='YS')})
        time_station['SOS'] = time_station['SOS'] + time_station['Referenzjahr'] + time_station['time'] 
        if not(self.latlon_proj):
            time_station['SOS'] += pd.Timedelta(12, 'h')
        time_station = time_station.rename({'time':'time_from_SOS', 'SOS':'time'})
        time_station = time_station.reset_index('Stations_id').reset_coords(names = 'Stations_id')
        self.time_station = time_station
        self.driver_data_for_ML = self.GDD_driver_data[driver_variable].sel(time_station)
        self.driver_data_for_ML = self.driver_data_for_ML.rename({'index': 'Stations_id'})
        self.driver_data_for_ML = self.driver_data_for_ML.set_xindex(['Stations_id'])
        self.driver_data_for_ML['Referenzjahr'] = pd.to_datetime(self.driver_data_for_ML['Referenzjahr']).year
        self.driver_frame_for_ML = self.driver_data_for_ML.to_dataframe(dim_order = ['Referenzjahr', 'Stations_id', 'time_from_SOS'])
        self.driver_frame_for_ML = pd.concat([self.driver_frame_for_ML[driver_variable].unstack(),
                                            self.driver_frame_for_ML['lat'].unstack()['0 days'].rename('lat'),
                                            self.driver_frame_for_ML['lon'].unstack()['0 days'].rename('lon'),
                                            self.driver_frame_for_ML['time'].unstack()['0 days'].rename('WC SOS')], axis=1)
        self.driver_frame_for_ML.rename(columns={self.driver_frame_for_ML.columns[x]: f'{driver_variable} day {x}' for x in range(200)}, inplace=True)
        self.driver_frame_for_ML = pd.merge(self.driver_frame_for_ML.reset_index(), self.ds_observed, how='left', on=['Referenzjahr', 'Stations_id'], suffixes=(None, '_observed')).drop(['lat_observed', 'lon_observed'], axis = 1)
        if self.phase_dates_calculated:
            self.driver_frame_for_ML = pd.merge(self.driver_frame_for_ML, self.phase_dates_array.reset_index(), how='left', on=['Referenzjahr', 'Stations_id'])

    def subsample_X_y(self, subsample_frac = 0.5):
        self.subsample = np.random.choice(np.arange(self.y_for_ML.shape[0]),np.int64(np.floor(self.y_for_ML.shape[0]*subsample_frac)))
        self.training_X = self.X_for_ML[self.subsample, :]
        self.training_y = self.y_for_ML[self.subsample, :]
        self.complement_of_subsample = np.delete(np.arange(self.y_for_ML.shape[0]), self.subsample)
        self.verification_X = self.X_for_ML[self.complement_of_subsample, :]
        self.verification_y = self.y_for_ML[self.complement_of_subsample, :]

        self.training_referenzjahr = self.GDD_driver_data['Referenzjahr'].values[self.subsample]
        self.training_stationid = self.GDD_driver_data['Stations_id'].values[self.subsample]
        self.verification_referenzjahr = self.GDD_driver_data['Referenzjahr'].values[self.complement_of_subsample]
        self.verification_stationid = self.GDD_driver_data['Stations_id'].values[self.complement_of_subsample]
    
    def decision_tree(self, md=20):
        self.regr = tree.DecisionTreeRegressor(max_depth=md, min_samples_leaf=5)
        self.fit = self.regr.fit(self.training_X, self.training_y)
        data_ML_training = {'Stations_id': np.int64(self.GDD_driver_data['Stations_id'].values[self.subsample]),
                        'Referenzjahr': np.int64(self.GDD_driver_data['Referenzjahr'].values[self.subsample]),
                        'Training': np.array([True for count in range(len(self.subsample))])
                        }
        data_ML_verification = {'Stations_id': np.int64(self.GDD_driver_data['Stations_id'].values[self.complement_of_subsample]),
                        'Referenzjahr': np.int64(self.GDD_driver_data['Referenzjahr'].values[self.complement_of_subsample]),
                        'Training': np.array([False for count in range(len(self.complement_of_subsample))])
                        }
        self.ds_ML_predictions_training = pd.DataFrame(data_ML_training)
        self.ds_ML_predictions_verification = pd.DataFrame(data_ML_verification)
        #Add modelled phase dates etc. to the comparison set.
        for phase_index, phase in enumerate(self.phase_list):
            self.ds_ML_predictions_training[f'ML prediction emergence to {phase}'] = self.fit.predict(self.training_X)[:, phase_index]
            self.ds_ML_predictions_verification[f'ML prediction emergence to {phase}'] = self.fit.predict(self.verification_X)[:, phase_index]
            self.ds_ML_predictions_training[f'ML check obs to {phase}'] = self.training_y[:, phase_index]
            self.ds_ML_predictions_verification[f'ML check obs to {phase}'] = self.verification_y[:, phase_index]
        self.ds_ML_predictions_training = self.ds_ML_predictions_training.drop_duplicates()
        self.ds_ML_predictions_verification = self.ds_ML_predictions_verification.drop_duplicates()
        self.ds_ML_results = pd.concat([self.ds_ML_predictions_verification, self.ds_ML_predictions_training], axis=0)
        self.ds_ML_results.set_index(['Referenzjahr', 'Stations_id'], inplace=True)
    
    def ML_modelled_observed(self):
        self.ds_ML_modelled_observed = pd.concat([self.ds_ML_results, self.ds_comparison, self.ds_observed], axis = 1)

C:\Users\wlwc1989\AppData\Local\Temp\ipykernel_36212\2307955636.py:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  station_data1 = pd.read_csv("https://opendata.dwd.de/climate_environment/CDC/help/PH_Beschreibung_Phaenologie_Stationen_Jahresmelder.txt",sep = ";\s+|;\t+|;\s+\t+|;\t+\s+|;|\s+;|\t+;|\s+\t+;|\t+\s+;", encoding='cp1252', on_bad_lines='skip')
C:\Users\wlwc1989\AppData\Local\Temp\ipykernel_36212\2307955636.py:5: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  station_data2 = pd.read_csv("https://opendata.dwd.de/climate_environment/CDC/help/PH_Beschreibung_Phaenologie_Stationen_Sofortmelder.txt"

In [4]:
Maize_set = Phenology_set('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\maize_phenology2_full_june_2025.csv', 
                           raw=False, full_ds = True, already_located = True)
Maize_set.drop_columns(['Unnamed: 9', 'Unnamed: 0'])
Maize_set.phase_order_name([10, 12, 67, 65, 5, 6, 19, 20, 21, 24, ])

Column Unnamed: 9 not found


In [ ]:
phen = Maize_set.ds_observed.drop_duplicates(subset = Maize_set.identifying_subset).dropna(subset = ['observed time to beginning of flowering'])
phen_known = phen.loc[phen['SORTE'] != ' Mais, Sorte unbekannt'].dropna(subset = ['SORTE'])
print(len(phen_known)/len(phen))

In [9]:
#Maize_set = Phenology_set("C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\PH_Jahresmelder_Landwirtschaft_Kulturpflanze_Mais_1936_2023_hist.txt", raw = True)
#Maize_set = Phenology_set('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\maize_phenology_20250224.csv', raw = False)
Maize_set = Phenology_set('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\maize_phenology2_full_june_2025.csv', 
                           raw=False, full_ds = True, already_located = True)
Maize_set.drop_columns(['Unnamed: 9', 'Unnamed: 0'])
Maize_set.phase_order_name([10, 12, 67, 65, 5, 6, 19, 20, 21, 24, ])
Maize_set.phen_data['lat'] = Maize_set.phen_data['lat'].astype(float)
Maize_set.phen_data['lon'] = Maize_set.phen_data['lon'].astype(float)
#Maize_set.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\Germany_Pre_2000\\temp_vpd_DE_1991_2000.nc')
#Maize_set.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\ERA5_land2_2011_2024.nc') #'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\tas_hyras_5_1951_2020_v5-0_de.nc')#
#Maize_set.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\ERA5_DE_2001_2024_maxmin.nc')
#Maize_set.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\ERA5_DE_2001_2024_VPD2.nc')
Maize_set.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\ERA5_DE_2001_2024_VPD_fixed.nc')
Maize_set.make_input_array(latlon_proj=True, start_year=1991)
#Maize_set.GDD_driver_data = Maize_set.GDD_driver_data.where(Maize_set.GDD_driver_data['time'] >= np.datetime64('2012-01-01'), drop = True)
#Maize_set.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\ERA5_land2_2001_2010.nc')
Maize_set.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\ERA5_DE_1991_2000_VPD_fixed.nc')
Maize_set.make_input_array(latlon_proj=True)
Maize_set.phase_list = ['beginning of flowering', 'beginning of tilling sowing drilling'] #['beginning of emergence', 'beginning of flowering', 'yellow ripeness']
Maize_set.get_observed_dataset(pd_SOS = True, use_metadata=True)

Column Unnamed: 9 not found
2534
2534
project to new coords
interpolate driver to station locations


In [49]:
obs_to_use = Maize_set.ds_observed.drop_duplicates(subset = Maize_set.identifying_subset + ['SORTE']).dropna(subset = ['observed time to beginning of flowering'])

In [ ]:
obs_to_use = obs_to_use.loc[obs_to_use['WC SOS date'] >= np.datetime64('2001-01-01')]

In [12]:
vble = 'vpd'
Maize_set.get_X_y_for_ML2(driver_variable = vble, SOS_name = 'Planting date', start_year = 1991, end_year = 2024, drop_columns = ['observation', 'number'], numdays = 180)#, 'number', 'expver'

32769
Couldn't reset index for station
0
32769
1
32769
2
32769
3
32769
4
32769
5
32769
6
32769
7
32769
8
32769
9
32769
10
32769
11
32769
12
32769
13
32769
14
32769
15
32769
16
32769
17
32769
18
32769
19
32769
20
32769
21
32769
22
32769
23
32769
24
32769
25
32769
26
32769
27
32769
28
32769
29
32769
30
32769
31
32769
32
32769
33
32769
34
32769
35
32769
36
32769
37
32769
38
32769
39
32769
40
32769
41
32769
42
32769
43
32769
44
32769
45
32769
46
32769
47
32769
48
32769
49
32769
50
32769
51
32769
52
32769
53
32769
54
32769
55
32769
56
32769
57
32769
58
32769
59
32769
60
32769
61
32769
62
32769
63
32769
64
32769
65
32769
66
32769
67
32769
68
32769
69
32769
70
32769
71
32769
72
32769
73
32769
74
32769
75
32769
76
32769
77
32769
78
32769
79
32769
80
32769
81
32769
82
32769
83
32769
84
32769
85
32769
86
32769
87
32769
88
32769
89
32769
90
32769
91
32769
92
32769
93
32769
94
32769
95
32769
96
32769
97
32769
98
32769
99
32769
100
32769
101
32769
102
32769
103
32769
104
32769
105
32769
106
32769
1

In [18]:
Maize_set.driver_frame_for_ML.dropna(subset = 'observed time to beginning of flowering').to_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_ERA5_fixed_vpd.csv')

In [67]:
#Maize_set.driver_frame_for_ML.dropna(subset = ['observed time to beginning of flowering', f'{vble} at day 179'])['Referenzjahr'].min()

In [68]:
ds = Maize_set.driver_frame_for_ML.dropna(subset = ['observed time to beginning of flowering', f'{vble} at day 179'])
#ds.loc[:, [f'{vble} at day {n}' for n in range(180)]] += - 273.15
ds.to_csv(f'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_ERA5_{vble}_pd_SOS_2001_2024.csv')

In [3]:
phen_data_CIMMYT = pd.read_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\African_data\\with_SOS\\CIMMYT_phen_data.csv')

In [7]:
phen_data_CIMMYT = dataset_fctns.columns_to_datetime(phen_data_CIMMYT, ['PlantingDate', 'AnthesisDate', 'DaysToSilk'])
phen_data_CIMMYT = phen_data_CIMMYT.rename(columns={'AnthesisDate': 'observed time to beginning of flowering'})

In [13]:
phase ='beginning of flowering'
phen_data_CIMMYT['yrcode'] = phen_data_CIMMYT[f'observed time to {phase}'].dt.year
columns_to_keep = ['lat', 'lon', 'Stations_id', f'observed time to {phase}', 'yrcode', 'vargroup', 'Management', 'SOS', 'EOS', 'SOS2', 'EOS2', 'AEZ', 'PlantingDate']
phen_data_CIMMYT = phen_data_CIMMYT[columns_to_keep]#.groupby(['Stations_id', 'yrcode', 'lat', 'lon', 'vargroup', 'Management', ]).mean().reset_index()
phen_data_CIMMYT.drop_duplicates().dropna(subset = ['PlantingDate', 'observed time to beginning of flowering'])

,lat,lon,Stations_id,observed time to beginning of flowering,yrcode,vargroup,Management,SOS,EOS,SOS2,EOS2,AEZ,PlantingDate
0,-14.20,28.40,1041,2004-02-07 16:48:00,2004.0,EPOP,Optimal,314,148,8,128,1.0,2003-12-05
1,-14.20,28.40,1041,2004-02-08 04:48:00,2004.0,ILPO,Optimal,314,148,8,128,1.0,2003-12-05
2,-14.20,28.40,1041,2004-02-12 21:36:00,2004.0,EIHY,Optimal,314,148,8,128,1.0,2003-12-05
3,-14.20,28.40,1041,2004-02-10 19:12:00,2004.0,ILHY,Optimal,314,148,8,128,1.0,2003-12-05
4,-14.20,28.40,1041,2004-02-11 00:00:00,2004.0,ILHY,Optimal,314,148,8,128,1.0,2003-12-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26105,-24.49,34.75,737,2007-02-20 00:00:00,2007.0,EPOP,Optimal,260,159,46,167,2.0,2006-12-19
26106,-24.49,34.75,737,2007-02-15 07:12:00,2007.0,EPOP,Low N,260,159,46,167,2.0,2006-12-19
26107,-24.49,34.75,737,2007-02-18 02:24:00,2007.0,EPOP,Optimal,260,159,46,167,2.0,2006-12-19
26108,-24.49,34.75,737,2007-02-18 04:48:00,2007.0,EPOP,Low N,260,159,46,167,2.0,2006-12-19


In [22]:
phen_data_CIMMYT = pd.read_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\African_data\\with_SOS\\CIMMYT_phen_data.csv')
AEZ_data = pd.read_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\African_data\\Lobell2011\\EIL_site_latlon_with_AEZ.csv')
AEZ_data = AEZ_data.rename(columns={'LocationID':'Stations_id'}).drop('Unnamed: 0', axis=1)
phen_data_CIMMYT = phen_data_CIMMYT.merge(AEZ_data[['Stations_id', 'AEZ']], on='Stations_id', how='left')
phen_data_CIMMYT = dataset_fctns.columns_to_datetime(phen_data_CIMMYT, ['PlantingDate', 'AnthesisDate', 'DaysToSilk'])
phen_data_CIMMYT = phen_data_CIMMYT.rename(columns={'AnthesisDate': 'observed time to beginning of flowering'})


In [31]:
phase ='beginning of flowering'
phen_data_CIMMYT['yrcode'] = phen_data_CIMMYT[f'observed time to {phase}'].dt.year
columns_to_keep = ['lat', 'lon', 'Stations_id', f'observed time to {phase}', 'yrcode', 'vargroup', 'Management', 'SOS', 'EOS', 'SOS2', 'EOS2', 'AEZ', 'PlantingDate']
phen_data_CIMMYT = phen_data_CIMMYT[columns_to_keep]#.groupby(['Stations_id', 'yrcode', 'lat', 'lon', 'vargroup', 'Management', ]).mean().reset_index()
phen_data_CIMMYT['WC SOS date'] = pd.to_datetime(phen_data_CIMMYT['PlantingDate'].dt.date)
phen_data_CIMMYT['Referenzjahr'] = phen_data_CIMMYT['yrcode']
phen_data_CIMMYT = phen_data_CIMMYT.loc[(phen_data_CIMMYT['observed time to beginning of flowering'] - phen_data_CIMMYT['WC SOS date']).dt.days < 240]
phen_data_CIMMYT['identifier'] = phen_data_CIMMYT.index

In [40]:
vble = 'ssrd_tp'
vble='vpd'
vble = 'temp'
Maize_set_Africa = Phenology_set('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\African_data\\Lobell2011\\maizedata.lobell.sep2011.csv', raw = False, dwd_data=False)
Maize_set_Africa.phen_data = phen_data_CIMMYT
Maize_set_Africa.phase_list = ['beginning of flowering']
Maize_set_Africa.identifying_subset = ['Stations_id', 'lat', 'lon', 'Management', 'vargroup', 'Referenzjahr', 'identifier']
Maize_set_Africa.get_mean_T(f'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\Africa\\{vble}_SSA_1999_2008.nc')#_fixed.nc
Maize_set_Africa.phen_data['Eintrittsdatum'] = Maize_set_Africa.phen_data['PlantingDate']
#Maize_set_Africa.phen_data['Stations_id'] = Maize_set_Africa.phen_data['sitecode']
Maize_set_Africa.make_input_array(latlon_proj=True, start_year = 1999)
Maize_set_Africa.ds_observed = Maize_set_Africa.phen_data[['identifier', 'Stations_id', 'yrcode', 'lat', 'lon', 'vargroup', 'Management', 'observed time to beginning of flowering', 'WC SOS date']]
flowering_times = Maize_set_Africa.ds_observed.loc[:, 'observed time to beginning of flowering'] - Maize_set_Africa.ds_observed.loc[:, 'WC SOS date']
Maize_set_Africa.ds_observed = Maize_set_Africa.ds_observed.drop('observed time to beginning of flowering', axis=1)#[:, 'observed time to beginning of flowering'] = pd.to_timedelta(0, 'D')
Maize_set_Africa.ds_observed.loc[:, 'observed time to beginning of flowering'] = flowering_times
Maize_set_Africa.ds_observed.rename(columns={'yrcode':'Referenzjahr'}, inplace=True)
Maize_set_Africa.metadata_cols = []

111
111
project to new coords
interpolate driver to station locations


In [41]:
for vble in ['t2m', 't2max', 't2min']:#['ssrd', 'tp']:#['vpd']:#
    print(vble)
    #Maize_set_Africa.get_X_y_for_ML(driver_variable = vble, SOS_name = 'WC SOS date', start_year = 1999, end_year = 2008, drop_columns = ['number', 'observation'])
    Maize_set_Africa.get_X_y_for_ML2(driver_variable = vble, SOS_name = 'WC SOS date', start_year = 1999, end_year = 2008, drop_columns = ['number', 'observation'], numdays = 200)
    ds = Maize_set_Africa.driver_frame_for_ML.dropna(subset = ['observed time to beginning of flowering', f'{vble} at day 199'])
    ds.to_csv(f'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_SSA_{vble}_huge_set.csv')

t2m
22792
Couldn't reset index for station
0
22792
1
22792
2
22792
3
22792
4
22792
5
22792
6
22792
7
22792
8
22792
9
22792
10
22792
11
22792
12
22792
13
22792
14
22792
15
22792
16
22792
17
22792
18
22792
19
22792
20
22792
21
22792
22
22792
23
22792
24
22792
25
22792
26
22792
27
22792
28
22792
29
22792
30
22792
31
22792
32
22792
33
22792
34
22792
35
22792
36
22792
37
22792
38
22792
39
22792
40
22792
41
22792
42
22792
43
22792
44
22792
45
22792
46
22792
47
22792
48
22792
49
22792
50
22792
51
22792
52
22792
53
22792
54
22792
55
22792
56
22792
57
22792
58
22792
59
22792
60
22792
61
22792
62
22792
63
22792
64
22792
65
22792
66
22792
67
22792
68
22792
69
22792
70
22792
71
22792
72
22792
73
22792
74
22792
75
22792
76
22792
77
22792
78
22792
79
22792
80
22792
81
22792
82
22792
83
22792
84
22792
85
22792
86
22792
87
22792
88
22792
89
22792
90
22792
91
22792
92
22792
93
22792
94
22792
95
22792
96
22792
97
22792
98
22792
99
22792
100
22792
101
22792
102
22792
103
22792
104
22792
105
22792
106
227

In [45]:
phen_data_CIMMYT = dataset_fctns.prepare_African_phen_ds(phen_data_CIMMYT, 'beginning of flowering', reducer = 'max')#'PlantingDate', 
phen_data_CIMMYT = phen_data_CIMMYT.loc[(phen_data_CIMMYT['observed time to beginning of flowering'] - phen_data_CIMMYT['WC SOS date']).dt.days < 240]
phen_data_CIMMYT['WC SOS date'] = phen_data_CIMMYT['PlantingDate']
phen_data_CIMMYT['Referenzjahr'] = phen_data_CIMMYT['yrcode']

yes


In [9]:
phen_data_CIMMYT['time to flowering'] = phen_data_CIMMYT['observed time to beginning of flowering'] - phen_data_CIMMYT['PlantingDate']

In [14]:
phen_data_CIMMYT = pd.read_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\African_data\\with_SOS\\CIMMYT_phen_data.csv')
phen_data_CIMMYT = dataset_fctns.columns_to_datetime(phen_data_CIMMYT, ['PlantingDate', 'AnthesisDate', 'DaysToSilk'])
phen_data_CIMMYT = phen_data_CIMMYT.rename(columns={'AnthesisDate': 'observed time to beginning of flowering'})
phen_data_CIMMYT = dataset_fctns.prepare_African_phen_ds(phen_data_CIMMYT, 'beginning of flowering', reducer = 'max')#'PlantingDate', 
phen_data_CIMMYT = phen_data_CIMMYT.loc[(phen_data_CIMMYT['observed time to beginning of flowering'] - phen_data_CIMMYT['WC SOS date']).dt.days < 240]
phen_data_CIMMYT['WC SOS date'] = phen_data_CIMMYT['PlantingDate']
phen_data_CIMMYT['Referenzjahr'] = phen_data_CIMMYT['yrcode']

yes


In [32]:
phen_data_after_97 = Maize_set_Africa.phen_data.loc[Maize_set_Africa.phen_data['WC SOS date'] >= np.datetime64('1999-01-01')]
ds_t2m = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 't2m', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)
ds_t2max = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 't2max', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)
ds_t2min = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 't2min', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)

#ds1 = ds1.dropna(subset = ['temperature at day 0', 'observed time to yellow ripeness']).drop_duplicates()#
for ds_index, ds in enumerate([ds_t2m, ds_t2max, ds_t2min]):#, ds_vpd]:
    ds_name = ['t2m', 't2max', 't2min'][ds_index]
    ds['observed time to beginning of flowering'] = ds['observed time to beginning of flowering'] - ds['WC SOS date']
    ds.to_csv(f'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_Africa_{ds_name}_PD_SOS4.csv')

0     2002-12-17
1     2002-12-13
3     2004-02-21
4     2004-02-20
5     2004-02-20
         ...    
688   2002-11-29
689   2005-12-16
690   2005-12-16
691   2002-11-29
704   2004-12-24
Name: WC SOS date, Length: 649, dtype: datetime64[ns]
Couldn't reset index for station
0


KeyError: 'Management'

In [131]:
Maize_set_Africa = Phenology_set('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\African_data\\Lobell2011\\maizedata.lobell.sep2011.csv', raw = False, dwd_data=False)
Maize_set_Africa.phen_data = phen_data_CIMMYT
Maize_set_Africa.phase_list = ['beginning of flowering']
Maize_set_Africa.identifying_subset = ['Stations_id', 'lat', 'lon', 'Management', 'vargroup', 'Referenzjahr']
Maize_set_Africa.get_mean_T('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\Saved_files\\ERA5\\Africa\\vpd_SSA_1999_2008.nc')
Maize_set_Africa.phen_data['Eintrittsdatum'] = Maize_set_Africa.phen_data['PlantingDate']
#Maize_set_Africa.phen_data['Stations_id'] = Maize_set_Africa.phen_data['sitecode']
Maize_set_Africa.make_input_array(latlon_proj=True, start_year = 1999)
Maize_set_Africa.ds_observed = Maize_set_Africa.phen_data[['Stations_id', 'yrcode', 'lat', 'lon', 'vargroup', 'Management', 'observed time to beginning of flowering', 'WC SOS date']]
flowering_times = Maize_set_Africa.ds_observed.loc[:, 'observed time to beginning of flowering'] - Maize_set_Africa.ds_observed.loc[:, 'WC SOS date']
Maize_set_Africa.ds_observed = Maize_set_Africa.ds_observed.drop('observed time to beginning of flowering', axis=1)#[:, 'observed time to beginning of flowering'] = pd.to_timedelta(0, 'D')
Maize_set_Africa.ds_observed.loc[:, 'observed time to beginning of flowering'] = flowering_times
Maize_set_Africa.ds_observed.rename(columns={'yrcode':'Referenzjahr'}, inplace=True)
Maize_set_Africa.metadata_cols = []

108
108
project to new coords
interpolate driver to station locations


In [132]:
for vble in ['vpd']:#['ssrd', 'tp']:#['t2m', 't2max', 't2min']:
    print(vble)
    #Maize_set_Africa.get_X_y_for_ML(driver_variable = vble, SOS_name = 'WC SOS date', start_year = 1999, end_year = 2008, drop_columns = ['number', 'observation'])
    Maize_set_Africa.get_X_y_for_ML2(driver_variable = vble, SOS_name = 'WC SOS date', start_year = 1999, end_year = 2008, drop_columns = ['number', 'observation'])
    ds = Maize_set_Africa.driver_frame_for_ML.dropna(subset = ['observed time to beginning of flowering', f'{vble} at day 300'])
    ds.to_csv(f'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_SSA_{vble}_varieties.csv')

vpd
655
Couldn't reset index for station
0
655
1
655
2
655
3
655
4
655
5
655
6
655
7
655
8
655
9
655
10
655
11
655
12
655
13
655
14
655
15
655
16
655
17
655
18
655
19
655
20
655
21
655
22
655
23
655
24
655
25
655
26
655
27
655
28
655
29
655
30
655
31
655
32
655
33
655
34
655
35
655
36
655
37
655
38
655
39
655
40
655
41
655
42
655
43
655
44
655
45
655
46
655
47
655
48
655
49
655
50
655
51
655
52
655
53
655
54
655
55
655
56
655
57
655
58
655
59
655
60
655
61
655
62
655
63
655
64
655
65
655
66
655
67
655
68
655
69
655
70
655
71
655
72
655
73
655
74
655
75
655
76
655
77
655
78
655
79
655
80
655
81
655
82
655
83
655
84
655
85
655
86
655
87
655
88
655
89
655
90
655
91
655
92
655
93
655
94
655
95
655
96
655
97
655
98
655
99
655
100
655
101
655
102
655
103
655
104
655
105
655
106
655
107
655
108
655
109
655
110
655
111
655
112
655
113
655
114
655
115
655
116
655
117
655
118
655
119
655
120
655
121
655
122
655
123
655
124
655
125
655
126
655
127
655
128
655
129
655
130
655
131
655
132
655
133
6

In [110]:
def get_X_y_for_ML2(self, driver_variable = 't2m', start_year = 2001, end_year = 2025, drop_columns = ['number', 'lon', 'lat', 'observation'], SOS_name = 'WC SOS date', numdays = 220):
    self.observations_to_use = self.ds_observed[['Stations_id', 'Referenzjahr', SOS_name] + self.metadata_cols].loc[(self.ds_observed[SOS_name] >= np.datetime64(f'{start_year}-01-01'))*(self.ds_observed[SOS_name] <= np.datetime64(f'{end_year + 1}-01-01'))].dropna(how='all').drop_duplicates(subset = ['Referenzjahr', 'Stations_id'])#subset = ['Referenzjahr', 'Stations_id']
    #print(self.observations_to_use, self.observations_to_use.drop_duplicates(subset = ['Stations_id', 'Planting date']))
    #self.observations_to_use  = self.observations_to_use.loc[self.observations_to_use[SOS_name].dt.year == self.observations_to_use['Referenzjahr']]
    print(len(self.observations_to_use))#, self.observations_to_use.drop_duplicates(['Referenzjahr', 'Stations_id']))
    # make an indexing array to pull values from the array of temperatures
    time_station = xr.Dataset.from_dataframe(self.observations_to_use)
    time_station = time_station.rename({'index':'observation', SOS_name:'time'})
    #print(time_station)
    if not(self.latlon_proj):
        time_station['time'] += np.timedelta64(12, 'h')# - np.timedelta64(60, 'D')

    ## Initiate development time storage object - a list with a value for all the stations, that will change over time and be stored in a list.
    t_dev = np.zeros(time_station.sizes['observation']) #Continuous development time. When this passes through some thresholds then have change in phase.
    dev_time_series = [t_dev.copy()]
    ## Make sure driver dataset uses station id to index this dimension
    try:
        self.GDD_driver_data = self.GDD_driver_data.set_xindex(['Stations_id'])
    except:
        print('Couldn\'t reset index for station')
    #print(time_station)
    #print(time_station[['observation', 'time']].drop_duplicates('observation'))
    #Run model
    first = True
    for day in range(310):
        # Pull values for temperature out of data frame
        print(day)
        #print(time_station['Referenzjahr'])
        driver_values = self.GDD_driver_data.sel(time_station[['Stations_id', 'time']])#[driver_variable]#.values
        driver_values['Referenzjahr'] = time_station['Referenzjahr']
        driver_frame_at_day = driver_values[[driver_variable, 'Stations_id', 'Referenzjahr', 'time']].to_pandas().reset_index().drop(drop_columns, axis=1)
        if first:
            years = driver_frame_at_day['time'].dt.year
            first = False
        #driver_frame_at_day['Referenzjahr'] = years #driver_frame_at_day['time'].dt.year
        #driver_frame_at_day.loc[driver_frame_at_day['time'].dt.dayofyear < 80, 'Referenzjahr'] = driver_frame_at_day.loc[driver_frame_at_day['time'].dt.dayofyear < 50, 'time'] .dt.year - 1
        #print(driver_frame_at_day)
        driver_frame_at_day = driver_frame_at_day.drop('time', axis=1)
        driver_frame_at_day = driver_frame_at_day.rename(columns = {driver_variable:f'{driver_variable} at day {day}'})
        #print(driver_frame_at_day)
        #print(driver_frame_at_day.drop_duplicates(subset = ['Referenzjahr', 'Stations_id']))
        print(len(self.observations_to_use))
        self.observations_to_use = self.observations_to_use.merge(driver_frame_at_day, on=['Referenzjahr', 'Stations_id']).dropna(how='all')
        time_station['time'] += np.timedelta64(1, 'D')
    self.driver_frame_for_ML = self.observations_to_use.merge(self.ds_observed[['Referenzjahr', 'Stations_id'] + [f'observed time to {phase}' for phase in self.phase_list]]).drop_duplicates(subset = ['Referenzjahr', 'Stations_id'])
    return self

In [63]:
SOS_name = 'Planting date'
start_year = 2011
end_year = 2024
Maize_set.observations_to_use = Maize_set.ds_observed[['Stations_id', 'Referenzjahr', SOS_name]].where((Maize_set.ds_observed['Referenzjahr'] >= start_year)*(Maize_set.ds_observed['Referenzjahr'] <= end_year)).dropna().drop_duplicates(subset = ['Referenzjahr', 'Stations_id'])

In [20]:
Maize_set.driver_frame_for_ML.to_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_ERA5_t2min_pd_SOS.csv')

In [ ]:
def put_temp_values_in_frame(driver_array, ds_observed, driver_variable, 
                             latlon_proj = True, phase_list = ['yellow ripeness'], 
                             SOS_offset = 0, station_locations = False,
                             identifying_subset = ['Stations_id', 'lat', 'lon', 'Management', 'vargroup'],
                             drop_columns = ['number', 'lon', 'lat', 'observation']):
    observations_to_use = ds_observed[identifying_subset + ['Referenzjahr', 'WC SOS date']].where(ds_observed['Referenzjahr'] > 1999).dropna(how='all').drop_duplicates()#['Stations_id', 'Referenzjahr', 'WC SOS date']
    observations_to_use['WC SOS date'] += np.timedelta64(SOS_offset, 'D')
    print(observations_to_use['WC SOS date'])
    observations_to_use['SOS_year'] = observations_to_use['WC SOS date'].dt.year
    observations_to_use['WC SOS date'] = pd.to_datetime(observations_to_use['WC SOS date'].dt.date)
    observations_to_use = observations_to_use.drop_duplicates(subset = ['SOS_year', 'Stations_id'])
    # make an indexing array to pull values from the array of temperatures
    time_station = xr.Dataset.from_dataframe(observations_to_use)
    time_station = time_station.rename({'index':'observation', 'WC SOS date':'time'})
    #print(time_station)
    if not(latlon_proj):
        time_station['time'] += np.timedelta64(12, 'h')
    #print(time_station['time'].min())
    time_vals = time_station['time'].values
    #print(time_vals[~np.isin(time_vals, driver_array['time'].values)])
    ## Initiate development time storage object - a list with a value for all the stations, that will change over time and be stored in a list.
    t_dev = np.zeros(time_station.sizes['observation']) #Continuous development time. When this passes through some thresholds then have change in phase.
    dev_time_series = [t_dev.copy()]
    ## Make sure driver dataset uses station id to index this dimension
    try:
        driver_array = driver_array.set_xindex(['Stations_id'])
    except:
        print('Couldn\'t reset index for station')
    
    #Run model
    for day in range(300):
        print(day)
        # Pull values for temperature out of data frame
        driver_values = driver_array.sel(time_station[['Stations_id', 'time']])#[driver_variable]#.values 
        #print('sel function applied')
        driver_frame_at_day = driver_values[[driver_variable, 'time'] + identifying_subset].to_pandas().reset_index().drop(drop_columns, axis=1)#'Stations_id', 
        #print('converted to pandas frame')
        if day == 0:
            SOS_years = driver_frame_at_day['time'].dt.year
            
            #Referenzjahrs = driver_frame_at_day['time'].dt.year + (driver_frame_at_day['time'].dt.dayofyear > 180)
        driver_frame_at_day['SOS_year'] = SOS_years #driver_frame_at_day['time'].dt.year
        #print(driver_frame_at_day)
        driver_frame_at_day = driver_frame_at_day.drop('time', axis=1)
        driver_frame_at_day = driver_frame_at_day.rename(columns = {driver_variable:f'{driver_variable} at day {day}'})
        #print(len(observations_to_use[['SOS_year', 'Stations_id']]), len(observations_to_use[['SOS_year', 'Stations_id']].drop_duplicates()),
        #    len(driver_frame_at_day[['SOS_year', 'Stations_id']]), len(driver_frame_at_day[['SOS_year', 'Stations_id']].drop_duplicates()))
        observations_to_use = observations_to_use.merge(driver_frame_at_day, on=identifying_subset + ['SOS_year'], how='inner')
        #print(observations_to_use)
        #print('merged')
        time_station['time'] += np.timedelta64(1, 'D')
    ds = observations_to_use.merge(ds_observed[['Referenzjahr', 'Stations_id'] + [f'observed time to {phase}' for phase in phase_list]]).drop_duplicates(subset = ['Referenzjahr', 'Stations_id'])
    #return ds
    ds = ds.dropna(subset = [f'{driver_variable} at day 0'] + [f'observed time to {phase}' for phase in phase_list]).drop_duplicates()#
    ds[[f'observed time to {phase}' for phase in phase_list]] = ds[[f'observed time to {phase}' for phase in phase_list]] + np.timedelta64(-SOS_offset, 'D')
    if type(station_locations) != bool:
        ds = get_station_locations(ds, station_locations)
    return ds#, observations_to_use, driver_frame_at_day

In [16]:
def get_X_y_for_ML2(self, driver_variable = 't2m', start_year = 2001, end_year = 2025, drop_columns = ['number', 'lon', 'lat', 'observation']):
    self.observations_to_use = self.ds_observed[['Stations_id', 'Referenzjahr', 'WC SOS date']].where((self.ds_observed['Referenzjahr'] >= start_year)*(self.ds_observed['Referenzjahr'] <= end_year)).dropna().drop_duplicates()
    # make an indexing array to pull values from the array of temperatures
    time_station = xr.Dataset.from_dataframe(self.observations_to_use)
    time_station = time_station.rename({'index':'observation', 'WC SOS date':'time'})
    #print(time_station)
    if not(self.latlon_proj):
        time_station['time'] += np.timedelta64(12, 'h')# - np.timedelta64(60, 'D')

    ## Initiate development time storage object - a list with a value for all the stations, that will change over time and be stored in a list.
    t_dev = np.zeros(time_station.sizes['observation']) #Continuous development time. When this passes through some thresholds then have change in phase.
    dev_time_series = [t_dev.copy()]
    ## Make sure driver dataset uses station id to index this dimension
    try:
        self.GDD_driver_data = self.GDD_driver_data.set_xindex(['Stations_id'])
    except:
        print('Couldn\'t reset index for station')
    
    #Run model
    for day in range(310):
        # Pull values for temperature out of data frame
        print(day)
        driver_values = self.GDD_driver_data.sel(time_station[['Stations_id', 'time']])#[driver_variable]#.values 
        driver_frame_at_day = driver_values[[driver_variable, 'Stations_id', 'time']].to_pandas().reset_index().drop(drop_columns, axis=1)
        driver_frame_at_day['Referenzjahr'] = driver_frame_at_day['time'].dt.year
        driver_frame_at_day = driver_frame_at_day.drop('time', axis=1)
        driver_frame_at_day = driver_frame_at_day.rename(columns = {driver_variable:f'{driver_variable} at day {day}'})
        print(driver_frame_at_day.columns.to_list())
        self.observations_to_use = self.observations_to_use.merge(driver_frame_at_day, on=['Referenzjahr', 'Stations_id'])
        time_station['time'] += np.timedelta64(1, 'D')
    self.driver_frame_for_ML = self.observations_to_use.merge(self.ds_observed[['Referenzjahr', 'Stations_id'] + [f'observed time to {phase}' for phase in self.phase_list]]).drop_duplicates(subset = ['Referenzjahr', 'Stations_id'])
    return self

In [25]:
Maize_set = get_X_y_for_ML2(Maize_set, driver_variable = 'tas', start_year = 2001, end_year = 2019, drop_columns=['lon', 'lat', 'observation', 'x', 'y'])

Couldn't reset index for station
0
['tas at day 0', 'Stations_id', 'Referenzjahr']
1
['tas at day 1', 'Stations_id', 'Referenzjahr']
2
['tas at day 2', 'Stations_id', 'Referenzjahr']
3
['tas at day 3', 'Stations_id', 'Referenzjahr']
4
['tas at day 4', 'Stations_id', 'Referenzjahr']
5
['tas at day 5', 'Stations_id', 'Referenzjahr']
6
['tas at day 6', 'Stations_id', 'Referenzjahr']
7
['tas at day 7', 'Stations_id', 'Referenzjahr']
8
['tas at day 8', 'Stations_id', 'Referenzjahr']
9
['tas at day 9', 'Stations_id', 'Referenzjahr']
10
['tas at day 10', 'Stations_id', 'Referenzjahr']
11
['tas at day 11', 'Stations_id', 'Referenzjahr']
12
['tas at day 12', 'Stations_id', 'Referenzjahr']
13
['tas at day 13', 'Stations_id', 'Referenzjahr']
14
['tas at day 14', 'Stations_id', 'Referenzjahr']
15
['tas at day 15', 'Stations_id', 'Referenzjahr']
16
['tas at day 16', 'Stations_id', 'Referenzjahr']
17
['tas at day 17', 'Stations_id', 'Referenzjahr']
18
['tas at day 18', 'Stations_id', 'Referenzjahr']

In [ ]:
phen_data_after_97 = Maize_set_Africa.phen_data.loc[Maize_set_Africa.phen_data['WC SOS date'] >= np.datetime64('1999-01-01')]
ds_t2m = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 't2m', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)
ds_t2max = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 't2max', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)
ds_t2min = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 't2min', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)

#ds1 = ds1.dropna(subset = ['temperature at day 0', 'observed time to yellow ripeness']).drop_duplicates()#
for ds_index, ds in enumerate([ds_t2m, ds_t2max, ds_t2min]):#, ds_vpd]:
    ds_name = ['t2m', 't2max', 't2min'][ds_index]
    ds['observed time to beginning of flowering'] = ds['observed time to beginning of flowering'] - ds['WC SOS date']
    ds.to_csv(f'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_Africa_{ds_name}_PD_SOS3.csv')
#ds['observed time to beginning of flowering'] = ds['observed time to beginning of flowering'] - ds['WC SOS date']
#ds = ds.where((ds['observed time to beginning of flowering'].dt.days > 0))# & (ds['observed time to beginning of flowering'].dt.days < 270)).dropna()
#ds.to_csv('C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_Africa_t2m_PD_SOS3.csv')
#ds = ds1
#ds1 = put_ERA5_in_array(ds1)

Couldn't reset index for station
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
2

In [9]:
phen_data_after_97 = Maize_set_Africa.phen_data.loc[Maize_set_Africa.phen_data['WC SOS date'] >= np.datetime64('1999-01-01')]
ds_ssrd = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 'ssrd', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)#, dad, otu 
ds_tp = put_temp_values_in_frame(Maize_set_Africa.GDD_driver_data, phen_data_after_97, 'tp', phase_list = ['beginning of flowering'],
                               SOS_offset=0, station_locations=False)#, dad, otu
for ds_index, ds in enumerate([ds_tp, ds_ssrd]):#, ds_vpd]:
    ds_name = ['tp', 'ssrd'][ds_index]
    ds['observed time to beginning of flowering'] = ds['observed time to beginning of flowering'] - ds['WC SOS date']
    ds.to_csv(f'C:\\Users\\wlwc1989\\Documents\\Phenology_Test_Notebooks\\phenology_dwd\\results_for_comparing\\Maize_ML_data_Africa_{ds_name}_PD_SOS3.csv')

Couldn't reset index for station
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
2